# Flat-guide GFMC for the transverse-field Ising model

The solver is `code/gfmc/`. This notebook is the record of the ground-state energy benchmark. Run it from the repository root, or from `notebook/`.

The operator on a stored bond list is

$$
H = -J \sum_{\langle ij \rangle} \sigma_i^z \sigma_j^z - \Gamma \sum_i \sigma_i^x - h \sum_i \sigma_i^z.
$$

In the $\sigma^z$ basis, $s_i = \pm 1$, the diagonal entry is $-J \sum s_i s_j - h \sum s_i$ and every single-spin flip has matrix element $-\Gamma$. The benchmark uses the ferromagnet $J = 1$.


## Projector

The run is continuous-time Green's function Monte Carlo with guiding function $\psi = 1$. A walker carries one configuration and a positive weight. It waits an exponential time of rate $\lambda = N\Gamma$, flips a site uniformly, and multiplies the weight over the wait $\Delta\tau$ by $\exp[-\Delta\tau (E_L - E_T)]$, where

$$
E_L(s) = E_{\mathrm{diag}}(s) - N\Gamma.
$$

Every block ends with systematic resampling. $E_T$ moves halfway toward the growth of the total weight. Two numbers estimate the ground-state energy: the mixed average of $E_L$, and the growth estimator. The class accepts another positive guide through `log_ratio`. This benchmark does not use one. `GFMC(..., h=None)` is the longitudinal field in the LTIM. `None` keeps the field stored on the operator. A number replaces that field and leaves the bonds unchanged.

`configurations` sets the initial walkers. Leave it empty for random product states, or pass `±1` spins of shape `(n_sites,)` or `(n_walkers, n_sites)`.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "code").exists():
    root = root.parent
sys.path.insert(0, str(root / "code"))

from gfmc import GFMC, iter_runs
from ising import chain, square

open_chain = chain(8, J=1.0, periodic=False)
periodic_chain = chain(8, J=1.0, periodic=True)
patch = square(3, 2, J=1.0, periodic=False)
print(open_chain.n_sites, "sites,", open_chain.bonds.shape[0], "open-chain bonds")
print(periodic_chain.bonds.shape[0], "periodic-chain bonds")
print(patch.n_sites, "sites,", patch.bonds.shape[0], "open 3×2 bonds")


In [ ]:
# One site is analytic: E = -sqrt(h^2 + Γ^2). This call is the API, not the saved benchmark.
one = GFMC(chain(1, h=0.0), gamma=1.0)
result = one.run(n_walkers=400, tau_block=0.2, n_equilibrate=10, n_measure=20, seed=1,
                 configurations=[1])
print(f"mixed {result['energy']:.6f} ± {result['energy_error']:.6f}")
print(f"growth {result['energy_growth']:.6f} ± {result['energy_growth_error']:.6f}")


## Saved benchmark

`python code/benchmark_tfim_gfmc.py` writes one file per run:

```text
data/tfim/gfmc/{lattice}/{boundary}/{size}/J{J}/gamma{gamma}/h{h}/seed{seed}_{id}.npz
```

`boundary` is `obc` or `pbc`. `size` is `n8` or `nx3_ny2`. A decimal point in a parameter is stored as `p`, so $\Gamma = 0.8$ is the directory `gamma0p8`. Kagome disorder, when regenerated, uses `data/kagome/{ensemble}/{obc|pbc}/L{L}_nr{N}.npz` and is not part of this table.

The reference column is exact diagonalization. Each GFMC error is an autocorrelation-aware standard error, with 2500 walkers. `id` is a hash of the run settings. Older files named `seed{seed}.npz` are previous runs and are still listed.


In [ ]:
runs = iter_runs(root / "data")
if not runs:
    print("No saved runs. From the repository root: python code/benchmark_tfim_gfmc.py")
else:
    header = f"{'lattice':<8} {'bc':<4} {'size':<9} {'Γ':>5} {'h':>5} {'exact':>10} {'mixed':>10} {'err':>8} {'growth':>10}"
    print(header)
    for run in runs:
        print(
            f"{run['lattice']:<8} {run['boundary']:<4} {run['size']:<9} "
            f"{run['gamma']:5.2f} {run['h']:5.2f} {run['exact_energy']:10.4f} "
            f"{run['energy']:10.4f} {run['energy_error']:8.4f} {run['energy_growth']:10.4f}"
        )


## Cluster jobs

Walkers in one population are not independent jobs. Inside a block each walker propagates on its own, but the block ends with one resampling that uses every weight, and $E_T$ is updated from the total weight. The next block starts from that shared population.

What does run in parallel:

- Separate populations. One job per seed, each with the full `n_walkers` already checked against exact diagonalization. Average the energies. The statistical error falls as $1/\sqrt{M}$ over $M$ jobs. Cutting one population into smaller populations increases the population-control bias, which scales as $1/N_w$.
- Different sizes, boundaries, $\Gamma$, and $h$. Each already has its own file and does not talk to the others.
- The hop loop inside one block can be split across cores, with the cores meeting at the resampling step. That requires a shared-memory or MPI change to the kernel. The present class does not do it.


## Scale

The target lattice has at least $10^3$ sites. The present kernel cannot reach that size, and changing it is not the current priority.

One block advances the population on a single core, walker after walker. The wall time scales as `n_walkers` times the imaginary time times the number of hops. The hop rate is $N\Gamma$. `ConstantGuide` draws the flipped site in constant time. Any other guide rebuilds all $N$ ratios on that hop. The number of blocks also grows, because the projection has to cover a time of order $1/\Delta$ and the gap $\Delta$ shrinks with $N$.

A run at $10^3$ sites needs the advances inside one block to run in parallel, with one reduction per block for the total weight, the shift $E_T$, and the resampling. That shared-memory or MPI change is deferred. Until it exists, the benchmark stays on the short chains and the $3\times 2$ square already in `data/tfim/gfmc/`.
